# Data Preprocessing

This notebook is the base for :
- Exploratory Data Analysis
- Data cleaning
- Data Impuation
- Feature Engineering

## Used Libraries

In [1]:
# type: ignore
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.impute import SimpleImputer
from scipy.stats import skew, kurtosis, binom
from sklearn.linear_model import LinearRegression

## Loading data

In [2]:
x_train = pd.read_csv('../x_train.csv', index_col='ID')
y_train = pd.read_csv('../y_train.csv', index_col='ID')
train = pd.concat([x_train, y_train], axis=1)
test = pd.read_csv('../x_test.csv', index_col='ID')
train.head()

,DATE,STOCK,INDUSTRY,INDUSTRY_GROUP,SECTOR,SUB_INDUSTRY,RET_1,VOLUME_1,RET_2,VOLUME_2,...,VOLUME_16,RET_17,VOLUME_17,RET_18,VOLUME_18,RET_19,VOLUME_19,RET_20,VOLUME_20,RET
ID,,,,,,,,,,,,,,,,,,,,,
0,0,2,18,5,3,44,-0.015748,0.147931,-0.015504,0.179183,...,0.630899,0.003254,-0.379412,0.008752,-0.110597,-0.012959,0.174521,-0.002155,-0.000937,True
1,0,3,43,15,6,104,0.003984,NaN,-0.090580,NaN,...,NaN,0.003774,NaN,-0.018518,NaN,-0.028777,NaN,-0.034722,NaN,True
2,0,4,57,20,8,142,0.000440,-0.096282,-0.058896,0.084771,...,-0.010336,-0.017612,-0.354333,-0.006562,-0.519391,-0.012101,-0.356157,-0.006867,-0.308868,False
3,0,8,1,1,1,2,0.031298,-0.429540,0.007756,-0.089919,...,0.012105,0.033824,-0.290178,-0.001468,-0.663834,-0.013520,-0.562126,-0.036745,-0.631458,False
4,0,14,36,12,5,92,0.027273,-0.847155,-0.039302,-0.943033,...,-0.277083,-0.012659,0.139086,0.004237,-0.017547,0.004256,0.579510,-0.040817,0.802806,False


The train and test inputs are composed of 46 features.

The target of this challenge is `RET` and corresponds to the fact that the **return is in the top 50% of highest stock returns**.

Since the median is very close to 0, this information should not change much with the idea to predict the sign of the return.

## Exploratory Data Analysis

In [ ]:
print(f"The train dataset contains {x_train.shape[0]} rows and {x_train.shape[1]} columns.")
print(f"The test dataset contains {test.shape[0]} rows and {test.shape[1]} columns.")
print(f'Features in the dataset: {x_train.columns}')

The dataset is made of 46 descriptive features: (all float / int values)

- `DATE`: an index of the date (the dates are randomized and anonymized so there is no continuity or link between any dates),
- `STOCK`: an index of the stock,
- `INDUSTRY`: an index of the stock industry domain (e.g., aeronautic, IT, oil company),
- `INDUSTRY_GROUP`: an index of the group industry,
- `SUB_INDUSTRY`: a lower level index of the industry,
- `SECTOR`: an index of the work sector,
- `RET_1` to `RET_20`: the historical residual returns among the last 20 days (i.e., `RET_1` is the return of the previous day and so on),
- `VOLUME_1` to `VOLUME_20`: the historical relative volume traded among the last 20 days (i.e., `VOLUME_1` is the relative volume of the previous day and so on),
The target variable: (binary)

- `RET`: the sign of the residual stock return at time t

Let's have a look on the structure of the data. In particular, I look at missing values, the balance of the dataset and potential correlation between features of the dataset.

In [ ]:
# Plotting missing values per features
missing_values = x_train.isnull().sum()
missing_values = missing_values[missing_values > 0]
missing_values = missing_values.sort_values(ascending=False)
missing_values.plot(kind='bar', figsize=(15, 5), title='Missing values per feature')


In [ ]:
# Plotting the distribution of the missing values per category
plt.figure(figsize=(15,10))
plt.subplots_adjust(hspace=0.5)
for i,category in enumerate(['INDUSTRY', 'INDUSTRY_GROUP', 'SECTOR', 'SUB_INDUSTRY', 'STOCK', 'DATE']): 
    plt.subplot(3,2,i+1)
    plt.title(category)
    plt.bar(train[category].sort_values().unique(),
            [(train[train[category]==sub_category].isna().sum(axis=1)>0).sum()/len(train[train[category]==sub_category])*100 for sub_category in train[category].sort_values().unique()])
    plt.xlabel('sub-category')
    plt.ylabel('%')
plt.show()

The plots show the distribution of NaN values per different type of category in percentage. We observe a "relatively" even distribution of NaN values for the categorical variable ``SECTOR``. However, the amount of NaN values for the other categorical variables appears to be less evenly distributed. It might be worthwhile to investigate if there are rows that predominantly consist of NaN values for the descriptive variables ``RET`` and ``VOLUME``. If such rows exist, we can drop them in good faith since these columns do not contribute to understanding the underlying structure. During this investigation, I noticed the following:

Given no observed returns, there is no volume observed. Therefore, we should only delete those observations where no return has been observed over the past days.

In [ ]:

ret_cols = [col for col in train.columns if 'RET_' in col]
volume_cols = [col for col in train.columns if 'VOLUME_' in col]

# describe the dataset
train[ret_cols + volume_cols].describe()

In [ ]:

# Plotting the distribution of the VOLUME features
sns.boxplot(train[[f'VOLUME_{day}' for day in range(1,21)]])
plt.ylim((-1.5,1))
plt.title('VOLUME features distribution')
plt.show()

# Plotting the distribution of the RET features
sns.boxplot(train[[f'RET_{day}' for day in range(1,21)]])
plt.ylim((-1.5,1))
plt.title('RET features distribution')
plt.show()

From the boxplots it becomes obvious that a median imputation for missing values in the colums is the better choice. (The mean is too optimistic).For the returns median and mean almost coincide. For simplicity we'll choose median imputation for both.

In [ ]:
# Target balance
plt.figure(figsize=(10,5))
plt.bar(y_train['RET'].value_counts().index, y_train['RET'].value_counts().values)
plt.title('RET target distribution')
plt.show()

# Correlation matrix with return, volume and target
ret_cols = [col for col in train.columns if 'RET' in col]
vol_cols = [col for col in train.columns if 'VOLUME' in col]
corr = train[ret_cols + vol_cols].corr().abs()
plt.figure(figsize=(10,10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation matrix with return and volume features')
plt.show()


The dataset is pretty balanced. We can therefore use a conventional approach, putting inbalance-related issues aside.
About correlation analysis : there seem to be no outlier or surprising values. We can therefore assume that apart from the NaN issue, the dataset requires only a minimum cleaning. Moreover, the original features seem to be uncorrelated.

## Data Preprocessing
- Cleaning : removing all rows with no observed returns over the past 5 days
- Imputation : Simple Impute the median for the remaining NaNs of RET_x and VOLUME_x


In [3]:
volume_cols = [col for col in train.columns if 'VOLUME_' in col]
ret_cols = [col for col in train.columns if 'RET_' in col]

In [4]:
# Drop all rows with too many missing returns over the 5 last days
ret_to_drop = train[(train[ret_cols[:5]].isna().sum(axis=1)/(train[ret_cols[:5]].shape[1]) >= 0.8)][ret_cols]
train.drop(index=ret_to_drop.index, inplace=True)

In [ ]:
# Impute missing values by taking the median of the feature of the SECTOR and DATE
for data in [train, test]:
    for col in data.columns:
        data[col] = data.groupby(['SECTOR', 'DATE'])[col].transform(lambda x: x.fillna(x.median()))

In [6]:
missing_values = train.isna().sum().sum(), test.isna().sum().sum()  
missing_values

(15, 8501)

In [7]:
# Simple imputer for the remaining missing values
imputer_train = SimpleImputer(strategy='median')
impuert_test = SimpleImputer(strategy='median')
train = pd.DataFrame(imputer_train.fit_transform(train), columns=train.columns, index=train.index)
test = pd.DataFrame(impuert_test.fit_transform(test), columns=test.columns, index=test.index)

# Check if there are still missing values
missing_values = train.isna().sum().sum(), test.isna().sum().sum()
missing_values

(0, 0)

## Feature Engineering

The main drawback in this challenge would be to deal with the noise. To do that, we could create some feature that aggregate features with some statistics. 

The following cell computes statistics on a given target conditionally to some features. For example, we want to generate a feature that describe the mean of `RET_1` conditionally to the `SECTOR` and the `DATE`.

**Ideas of improvement**: change shifts, the conditional features, the statistics, and the target. 

### Sectorial Aggregation

In [8]:
# supress warnings
import warnings
warnings.filterwarnings('ignore')

In [9]:
# Single Aggregation by DATE, STOCK, SECTOR
# Create new features based on the statistics by DATE aggregation for multiple shifts
statistics = {'mean':'mean', 'std':'std','skew': lambda x: skew(x, nan_policy='omit'), 'max':'max', 'min':'min'}

for target_feature in ['RET','VOLUME']:
    for gb_feature in ['DATE', 'STOCK', 'SECTOR']:
        for shift in tqdm([1,2,3,4,5], desc=f'{target_feature}_{gb_feature}'):
            for stat_name,stat in statistics.items():
                name = f'{target_feature}_{shift}_{gb_feature}_{stat_name}'
                feat = f'{target_feature}_{shift}'
                for data in [train, test]:
                    data[name] = data.groupby([gb_feature])[feat].transform(stat)


VOLUME_SECTOR: 100%|██████████| 5/5 [00:00<00:00,  8.08it/s]


In [11]:
# Dual Aggregation by DATE and SECTORIAL- LEVEL
# Create new features based on the statistics by multiple sector and date aggregation for multiple shifts

for target_feature in ['RET','VOLUME']:
    for [cat, date] in [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE'], ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']]:
        tmp_name = cat + '_' + date
        for shift in tqdm([1,2,3,4,5], desc=f'{target_feature}_{tmp_name}'):
            for stat_name,stat in statistics.items():
                name = f'{target_feature}_{shift}_{tmp_name}_{stat_name}'
                feat = f'{target_feature}_{shift}'
                for data in [train, test]:
                    data[name] = data.groupby([cat, date])[feat].transform(stat)

VOLUME_SUB_INDUSTRY_DATE: 100%|██████████| 5/5 [01:25<00:00, 17.12s/it]


In [14]:
# Create new features based on rolling statistics for the past weeks
weeks = 4

for target_feature in ['RET', 'VOLUME']:
    for week in tqdm(range(weeks), desc=f'{target_feature}_WEEK'):
        name = f'{target_feature}_WEEK_{week+1}'
        mean_name = 'mean_' + name
        std_name = 'std_' + name
        skew_name = 'skew_' + name
        # TODO : kurt_name = 'kurt_' + name
        for data in [train, test]:
            data[mean_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].mean(axis=1)
            data[std_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].std(axis=1)
            data[skew_name] = data[[f'{target_feature}_{week*5 + day}' for day in range(1,6)]].skew(axis=1)
        
# Normalize the mean of the return and volume features by the mean of the sector and date

for target_feature in ['RET', 'VOLUME']:
    for gb_features in [['SECTOR', 'DATE']]:
        tmp_name = '_'.join(gb_features)
        for shift in tqdm([1,2,3,4], desc=f'{target_feature}_{tmp_name}') :
            feat = f'mean_{target_feature}_WEEK_{shift}'
            name = f'mean_{target_feature}_WEEK_{shift}_/total_{feat}_{tmp_name}'
            for data in [train, test]:
                data[name] = data[feat]/data.groupby(gb_features)[feat].transform('sum')



VOLUME_SECTOR_DATE: 100%|██████████| 4/4 [00:00<00:00, 28.52it/s]


In [15]:
# Momentum features aggregation by sector and date

weeks = [1, 2, 3, 4]
targets = ['RET', 'VOLUME']

for target in targets:
    for week in tqdm(weeks, desc=f'{target}_Momentum'): 
        window_size = 5*week
        name = f'{target}_{window_size}_SECTOR_DATE_Momentum'
        for data in [train, test]:
            frame = data.copy()
            rolling_mean_target = frame.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(2, window_size+1)]].mean()
            target_1_mean = frame.groupby(by=['SECTOR', 'DATE'])[[f'{target}_1']].mean()
            target_1_mean_aligned, rolling_mean_target_aligned = target_1_mean.align(rolling_mean_target, axis=0, level='SECTOR')
            target_momentum = target_1_mean_aligned.sub(rolling_mean_target_aligned.mean(axis=1), axis=0)
            target_momentum.rename(columns={f'{target}_1': name},inplace=True)
            placeholder = frame.join(target_momentum, on=['SECTOR', 'DATE'], how='left')
            data[name] = placeholder[name] 

# RSI features aggregation by sector and date

targets = ["RET"]
window_size = [5, 10, 15, 20]

for window in tqdm(window_size, desc='RSI'):
    name = f"RSI_{window}_SECTOR_DATE"
    for target in targets:
        for data in [train, test]:
            avg_gain_sector_day = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1, window + 1)]].mean().agg(lambda x: x[x > 0].mean(), axis=1)
            avg_loss_sector_day = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1, window + 1)]].mean().agg(lambda x: x[x < 0].mean(), axis=1).abs()
            rs_sector_day = avg_gain_sector_day / avg_loss_sector_day
            rsi_sector_date = 100 - 100 / (1 + rs_sector_day)
            data[name] = data.join(rsi_sector_date.to_frame(name), on=['SECTOR', 'DATE'], how='left')[name]


# ADL features aggregation by sector and date

window_size = [5, 10, 15, 20]
for window in tqdm(window_size, desc='ADL'):
    name = f'ADL_{window}_SECTOR_DATE'
    for data in [train, test]:
        sum_adl = (data.groupby(by=["SECTOR", "DATE"])[[f'RET_{day}' for day in range(1, window + 1)]].apply(lambda x: (x > 0).sum()) - data.groupby(by=["SECTOR", "DATE"])[[f'RET_{day}' for day in range(1, window + 1)]].apply(lambda x: (x < 0).sum())).sum(axis=1)
        data[name] = data.join(sum_adl.to_frame(name), on=['SECTOR', 'DATE'], how='left')[name]


# Volatility features aggregation by sector and date

weeks = [1,2,3,4]
targets = ['RET', 'VOLUME']
   
for week in tqdm(weeks, desc='Volatility'): 
    window_size = 5*week
    for target in targets: 
        name = f'{target}_SECTOR_DATE_VOLATILITY_{window_size}'
        for data in [train, test]:
            rolling_std_target = data.groupby(by=['SECTOR', 'DATE'])[[f'{target}_{day}' for day in range(1,window_size+1)]].mean().std(axis=1).to_frame(name)
            placeholder = data.join(rolling_std_target, on=['SECTOR', 'DATE'], how='left')
            data[name] = placeholder[name]

Volatility: 100%|██████████| 4/4 [00:37<00:00,  9.45s/it]


### Temporal aggregation

In [18]:
def compute_moving_avg(df, cols):
    return df[cols].mean(axis=1, skipna=True)

def compute_volatility(df, cols):
    return df[cols].std(axis=1, skipna=True)

def compute_ema(df, cols, span=5):
    alpha = 2 / (span + 1)
    ema = df[cols[0]].copy()
    for col in cols[1:]:
        ema = alpha * df[col] + (1 - alpha) * ema
    return ema


def compute_momentum(df, col_start, col_end):
    return df[col_end] - df[col_start]

def compute_rsi(df, cols):
    gains = df[cols].clip(lower=0).mean(axis=1, skipna=True)
    losses = df[cols].clip(upper=0).abs().mean(axis=1, skipna=True)
    return 100 - (100 / (100 + (gains / losses)))

def compute_adl(df, cols):
    positive_counts = (df[cols] > 0).sum(axis=1, skipna=True)
    negative_counts = (df[cols] < 0).sum(axis=1, skipna=True)
    return positive_counts - negative_counts

def compute_likelihood(df, ret_cols):
    positive_counts = (df[ret_cols] > 0).sum(axis=1, skipna=True)
    n = len(ret_cols)
    p_hat = positive_counts / n  
    likelihood = binom.cdf(k=positive_counts+1, n=n+1, p=p_hat)
    
    return likelihood

def fit_ar_n_and_predict_ret(df, n):
    # Step 1: Fit AR(n) model on RET_1 = f(RET_2, ..., RET_(n+1))
    X_train = df[[f'RET_{i}' for i in range(2, n+2)]]  # Using n lags
    y_train = df['RET_1']  # Target variable

    ar_model = LinearRegression()
    ar_model.fit(X_train, y_train)

    # Step 2: Predict RET_0 using RET_1, ..., RET_n
    X_pred = df[[f'RET_{i}' for i in range(1, n+1)]].copy()  # Copy to avoid modification warnings
    X_pred.columns = X_train.columns  # Rename to match training features to avoid error

    X_pred.columns = X_train.columns 
    return ar_model.predict(X_pred)


# Z-Score Normalization
def z_score_normalization(df, cols, shift=1):
    mean = df[cols].mean(axis=1, skipna=True)
    std = df[cols].std(axis=1, skipna=True)
    column = cols[shift-1] if shift > 0 else cols[shift]
    return (df[column] - mean) / std


# Hurst Exponent
def compute_hurst(df, cols):

    def hurst_exponent(time_series):

        rescaled_range = []
        windows = []
        N = len(time_series)

        for n in range(2, N + 1):
            time_series_n = time_series[:n]
            # Step 1: Calculate the mean of the time series
            mean = np.mean(time_series_n)
            
            # Step 2: Calculate the cumulative sum
            cumulative_sum = np.cumsum(time_series_n - mean)

            # Step 3: Calculate the range (R) and standard deviation (S)
            R = np.max(cumulative_sum) - np.min(cumulative_sum)
            S = np.std(time_series_n)
            
            # Step 4: Rescaled range
            if S != 0:
                R_S = R / S
                rescaled_range.append(R_S)
                windows.append(n)

        y = np.log(rescaled_range)
        x = np.log(windows)

        mask = ~np.isnan(x) & ~np.isnan(y)
        x = x[mask]
        y = y[mask]

        # Step 5: Calculate the Hurst exponent by fitting a linear regression to the log-log plot
        if len(x) == 0:
            return np.nan
        else:
            hurst_exponent = LinearRegression().fit(x.reshape(-1, 1), y)
            H = hurst_exponent.coef_[0]
            return H
    
    tqdm.pandas(desc="Calculating Hurst Exponent")
    # Apply hurst_exponent for each row (representing a series of lagged returns for a stock)
    return df[cols].progress_apply(lambda row: hurst_exponent(row), axis=1)

# Volume-Return Interaction Features

# VWAR (Volume Weighted Average Return)
def compute_vwar(df, ret_cols, vol_cols):
    df_ret = df[ret_cols].values
    df_vol = df[vol_cols].values

    return (df_ret * df_vol).sum(axis=1) / df_vol.sum(axis=1)

# VoV (Volatility over Volume)
def compute_vov(df, ret_cols, vol_cols):
    df_ret = df[ret_cols].values
    df_vol = df[vol_cols].values
    
    return df_ret.std(axis=1) / df_vol.mean(axis=1)


In [ ]:
def generate_indicators(df, num_days = 20):
    ind_columns = {}

    feature_functions = {
        'MA': compute_moving_avg,
        'VOLATILITY': compute_volatility,
        'EMA': compute_ema,
        'MOMENTUM': compute_momentum,
        'RSI': compute_rsi,
        'ADL': compute_adl,
        'HURST': compute_hurst,
        'LIKELIHOOD': compute_likelihood,
        'VWAR' : compute_vwar,
        'VOV' : compute_vov
    }

    cols = {
        'RET': [f'RET_{i}' for i in range(1, num_days+1)],
        'VOLUME': [f'VOLUME_{i}' for i in range(1, num_days+1)]
    }
    
    for feature in ['RET', 'VOLUME']:
        ind_columns[f'ACF_{feature}'] = df[cols[feature]].apply(lambda x: np.corrcoef(x[:-1], x[1:])[0, 1], axis=1)
        ind_columns[f'MA_{feature}'] = feature_functions['MA'](df, cols[feature])
        ind_columns[f'VOLATILITY_{feature}'] = feature_functions['VOLATILITY'](df, cols[feature])
        ind_columns[f'EMA_{feature}'] = feature_functions['EMA'](df, cols[feature])
        ind_columns[f'MOMENTUM_{feature}'] = feature_functions['MOMENTUM'](df, cols[feature][0], cols[feature][-1])
        ind_columns[f'Z_SCORE_NORMALIZATION_{feature}_1'] = z_score_normalization(df, cols[feature], shift=1)
        ind_columns[f'Z_SCORE_NORMALIZATION_{feature}_2'] = z_score_normalization(df, cols[feature], shift=2)
        if feature == 'RET':
            ind_columns['RSI_RET'] = feature_functions['RSI'](df, cols[feature])
            ind_columns['ADL_RET'] = feature_functions['ADL'](df, cols[feature])
            ind_columns['LIKELIHOOD_RET'] = feature_functions['LIKELIHOOD'](df, cols[feature])
            ind_columns['RET_AR3_PRED'] = fit_ar_n_and_predict_ret(df, 3)
            ind_columns['HURST'] = feature_functions['HURST'](df, cols[feature])

    ind_columns['VWAR'] = feature_functions['VWAR'](df, cols['RET'], cols['VOLUME'])
    ind_columns['VOV'] = feature_functions['VOV'](df, cols['RET'], cols['VOLUME'])
    
    for shift in tqdm([4, 12, 20], desc="Calculating Features Rolling Statistics"):
        for feature in ['RET', 'VOLUME']:
            data = df[ cols[feature][:shift] ]
            # statistics
            ind_columns[f'Mean_{feature}_{shift}D'] = np.mean(data, axis=1)
            ind_columns[f'Std_{feature}_{shift}D'] = np.std(data, axis=1)
            ind_columns[f'Skew_{feature}_{shift}D'] = skew(data, nan_policy='omit', axis=1)
            ind_columns[f'Kurtosis_{feature}_{shift}D'] = kurtosis(data, nan_policy='omit', axis=1)
            if feature == 'RET':
                ind_columns[f'Range_{feature}_{shift}D'] = (lambda x: np.max(x, axis=1) - np.min(x, axis=1))(data)
                ind_columns[f'Momentum_{feature}_{shift}D'] = data.iloc[:, -1] - data.iloc[:, 0]
                ind_columns[f'Cumulative_{feature}_{shift}D'] = np.prod(1 + data, axis=1) - 1
            else:
                ind_columns[f'VOL_SURGE_{shift}D'] = (df['VOLUME_1'] - ind_columns[f'Mean_VOLUME_{shift}D']) / ind_columns[f'Std_VOLUME_{shift}D']
        
        # correlation between returns and volumes
        ret, vol = df[cols['RET'][:shift]], df[cols['VOLUME'][:shift]]
        ind_columns[f'Corr_RET_VOL_{shift}D'] = [np.corrcoef(ret.iloc[i], vol.iloc[i])[0, 1] for i in range(len(ret))]

    return ind_columns

In [20]:
ind_columns_train = generate_indicators(train)
ind_columns_test = generate_indicators(test)

Calculating Features Rolling Statistics: 100%|██████████| 3/3 [01:15<00:00, 25.19s/it]


In [22]:
df_train_indicators = pd.DataFrame(ind_columns_train)
df_test_indicators = pd.DataFrame(ind_columns_test)

In [29]:
train = pd.concat([train, df_train_indicators], axis=1)
test = pd.concat([test, df_test_indicators], axis=1)

### Additional Feature Aggregation and Sectorial Neutralization

In [32]:
statistics = {'mean':'mean', 'std':'std'}
for target_feature in tqdm(['HURST','VOV', 'VWAR', 'LIKELIHOOD_RET'], desc=f'Aggregating Features'):
    for [cat, date] in [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE']]: # TODO : ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']
        tmp_name = cat + '_' + date
        for stat_name,stat in statistics.items():
            name = f'{target_feature}_{tmp_name}_{stat_name}' 
            for data in [train, test]:
                data[name] = data.groupby([cat, date])[target_feature].transform(stat)

Aggregating Features: 100%|██████████| 4/4 [00:00<00:00,  5.79it/s]


In [33]:
for target_feature in ['RET', 'VOLUME']:
    for gb_features in [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE'], ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']]: 
        tmp_name = '_'.join(gb_features)
        for shift in tqdm([1,2,3,4], desc=f'{target_feature}_{tmp_name}_NEUTRAL') :
            feat = f'{target_feature}_{shift}'
            feat_mean = f'{target_feature}_{shift}_{tmp_name}_mean'
            feat_std = f'{target_feature}_{shift}_{tmp_name}_std'
            name = f'{target_feature}_{shift}_{tmp_name}_NEUTRAL'
            for data in [train, test]:
                data[name] = (data[feat] - data[feat_mean]) / data[feat_std]

for target_feature in tqdm(['HURST','VOV', 'VWAR', 'LIKELIHOOD_RET'], desc=f'Feature Neutralization'):
    for [cat, date] in [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE']]: # TODO : ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']
        tmp_name = cat + '_' + date
        for stat_name,stat in statistics.items():
            name = f'{target_feature}_{tmp_name}_NEUTRAL'
            feat_mean = f'{target_feature}_{tmp_name}_mean'
            feat_std = f'{target_feature}_{tmp_name}_std'
            for data in [train, test]:
                data[name] = (data[target_feature] - data[feat_mean]) / data[feat_std]


Feature Neutralization: 100%|██████████| 4/4 [00:00<00:00, 55.53it/s]


### Rank by sector date aggregation

In [34]:
# Rank RET and VOLUME features by SECTOR and DATE

for target_feature in ['RET','VOLUME']:
    for [cat, date] in [['SECTOR', 'DATE'], ['INDUSTRY_GROUP', 'DATE'], ['INDUSTRY', 'DATE'], ['SUB_INDUSTRY', 'DATE']]: # TODO : 
        tmp_name = cat + '_' + date
        for shift in tqdm(range(1,21), desc=f'{target_feature}_{tmp_name}_RANK'):
            name = f'{target_feature}_{shift}_{tmp_name}_RANK'
            feat = f'{target_feature}_{shift}'
            for data in [train, test]:
                data[name] = data.groupby([cat, date])[feat].rank(pct=True)


VOLUME_SUB_INDUSTRY_DATE_RANK: 100%|██████████| 20/20 [00:03<00:00,  5.10it/s]


## Data post-processing

In [35]:
infinites_train, infinites_test = train.isin([np.inf, -np.inf]).sum().sum(), test.isin([np.inf, -np.inf]).sum().sum()
infinites_train, infinites_test # Check for infinite values

(934, 70)

In [36]:
# infinite values
train.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

In [37]:
# NaN cols
nan_cols_train, nan_cols_test = train.isna().sum(), test.isna().sum()

In [38]:
nan_cols_train = nan_cols_train[nan_cols_train > 0]
nan_cols_test = nan_cols_test[nan_cols_test > 0]
nan_cols_train, nan_cols_test

(RET_1_STOCK_std                                32
 RET_1_STOCK_skew                               32
 RET_2_STOCK_std                                32
 RET_2_STOCK_skew                               32
 RET_3_STOCK_std                                32
                                              ... 
 VOV_INDUSTRY_GROUP_DATE_NEUTRAL               257
 VWAR_SECTOR_DATE_NEUTRAL                        9
 VWAR_INDUSTRY_GROUP_DATE_NEUTRAL               24
 LIKELIHOOD_RET_SECTOR_DATE_NEUTRAL              8
 LIKELIHOOD_RET_INDUSTRY_GROUP_DATE_NEUTRAL     23
 Length: 171, dtype: int64,
 RET_1_STOCK_std                               17
 RET_1_STOCK_skew                              17
 RET_2_STOCK_std                               17
 RET_2_STOCK_skew                              17
 RET_3_STOCK_std                               17
                                               ..
 VOV_INDUSTRY_GROUP_DATE_NEUTRAL                1
 VWAR_SECTOR_DATE_NEUTRAL                       1
 VWAR_INDUS

In [39]:

# Impute missing values by taking the median of the feature of the SECTOR and DATE
for data in [train, test]:
    for col in tqdm(data.columns, desc='Imputing Missing Values'):
        data[col] = data.groupby(['SECTOR', 'DATE'])[col].transform(lambda x: x.fillna(x.median()))

Imputing Missing Values: 100%|██████████| 728/728 [01:55<00:00,  6.30it/s]


In [41]:
train.isnull().sum().sum(), test.isnull().sum().sum()  # Checking there is no more missing values

(27985, 11763)

In [42]:
# Simple imputer for the remaining missing values
imputer_train = SimpleImputer(strategy='median')
impuert_test = SimpleImputer(strategy='median')
train = pd.DataFrame(imputer_train.fit_transform(train), columns=train.columns, index=train.index)
test = pd.DataFrame(impuert_test.fit_transform(test), columns=test.columns, index=test.index)

# Check if there are still missing values
missing_values = train.isna().sum().sum(), test.isna().sum().sum()
missing_values

(0, 0)

## Outputing the extended dataframe

In [43]:
# Save the preprocessed data in parquet
train.to_parquet('../train_extended.parquet')
test.to_parquet('../test_extended.parquet')

## Other Ideas

Next alpha factors to try :
- [https://arxiv.org/pdf/1601.00991](101 Formulaic Alphas) 
    - https://github.com/stefan-jansen/machine-learning-for-trading/blob/main/24_alpha_factor_library/03_101_formulaic_alphas.ipynb
    - https://github.com/sumilk/algo_trading/blob/main/3_formulaic_alphas.ipynb
    - List of Alpha using rets and vols :
        - Alpha 1
        - Alpha 8
        - Alpha 14
        - Alpha 25
        - Alpha 34
        - Alpha 56
        - Alpha 2
        - Alpha 18
        - Alpha 60

- Denoising
    - Wavelet Transform feature 
        - Haar
        - Daubechies
    - FFT (Fast Fourier Transform)
  
- Features on aggregated features
  - Hurst exponent computed on sectorial and date aggregated returns
  - Hurst with hurst library (require minimum 100 samples) by data augmentation of returns
- Conditional features on event triggering with apply method
- Deep dive in mean reverting, trend momentum indicator for return, volume, hybrid (potentially on aggregated features)
- Bench of all correlation between features and it's differents level aggregation features
  - correlation ``RET_X`` and ``RET_X_SECTOR_DATE``
  - correlation ``RET_X`` and ``RET_{X-1}_SECTOR_DATE``
  - Linear regression ``RET_X`` and ``RET_{X-1}_SECTOR_DATE``, ``RET_{X-2}_INDUSTRY_DATE``, etc
  - Previous using ``rank`` variant or normalisation